# Baseline: 30 Hz burst vs ARC-EX, three participants

**P03, P04 and NTA, cathodic, baseline** - no lidocaine, no vibration, all on **electrode 2**, so
the montage is the same for the three. The question is the same one in each: *does 30 Hz behave
differently from ARC-EX, and does it do so in all three people?*

## The rule this whole notebook follows

**Nothing in mA or mV is compared between participants.** Different anatomy, electrode placement,
impedance and EMG gain make raw numbers meaningless across people. Every figure normalises
*within* a participant first:

| | normalised to |
|---|---|
| intensity | that muscle's own motor threshold (`x MT`) |
| amplitude | that muscle's own biggest response in that recording (`%`) |

What is compared between people is then a **ratio** or a **shape**, never a raw number.

## What comes out

| | figure | the question it answers |
|---|---|---|
| 1 | threshold ratio | how much more current does ARC-EX need than 30 Hz, and is it the same factor in everyone? |
| 2 | normalised recruitment | once a muscle starts responding, does it grow more steeply under one protocol? |
| 3 | selectivity | when the current is raised, how much of the arm comes in at once? |
| 4 | depression along the train | does the response drop after the first pulse under 30 Hz but not ARC-EX, in all three? |

Motor thresholds come first, because everything above depends on them. They are detected
automatically and then **corrected by hand** in the picker, one cell per recording.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from functions import (set_style, load_run, pretty, waterfall,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       recruitment, fig_threshold_ratio, fig_recruitment, fig_selectivity)
from functions.paper import motor_thresholds, fig_depression, step_up
set_style()

## 1 · Config

In [ ]:
# ======== the six recordings ========
RUN = {   # (participant, protocol) -> csv, all cathodic baseline on electrode 2
    ("P03", "burst"): "tSCS_CHUV_data/15-07-2026/P03tscsHealthy/Burst_autosave_20260715_164431_737ms.csv",
    ("P03", "arcex"): "tSCS_CHUV_data/15-07-2026/P03tscsHealthy/Modulated_autosave_20260715_164630_593ms.csv",
    # ^^^ PLACEHOLDER - that session has 4 burst and 4 ARC-EX runs and no log file. These are the
    #     first matched pair after the vibration testing. Replace with the real baseline pair.
    ("P04", "burst"): "tSCS_CHUV_data/16-07-2026/P04tscsHealthy/Burst_autosave_20260716_142311_232ms.csv",
    ("P04", "arcex"): "tSCS_CHUV_data/16-07-2026/P04tscsHealthy/Modulated_autosave_20260716_142536_785ms.csv",
    ("NTA", "burst"): "tSCS_CHUV_data/24-07-2026/testSCS/Burst_autosave_20260724_102443_670ms.csv",
    ("NTA", "arcex"): "tSCS_CHUV_data/24-07-2026/testSCS/Modulated_autosave_20260724_103530_508ms.csv",
}
SUBJECTS  = ["P03", "P04", "NTA"]
PROTOCOLS = ["burst", "arcex"]
NAME  = {"burst": "30 Hz burst", "arcex": "ARC-EX"}
SCOL  = {"P03": "#1f3b73", "P04": "#e6550d", "NTA": "#2ca25f"}

# the muscles every one of the three recordings has, minus the proximal ones
MUSCLES = ["Thenar (L)", "Ext. digitorum (L)", "Flex. digitorum (L)", "Flex. carpi rad. (L)",
           "Thenar (R)", "Ext. digitorum (R)", "Flex. digitorum (R)", "Flex. carpi rad. (R)"]

# ---- peak detection: the same settings as every other notebook ----------------------------
N_PULSES, RESP_START_MS, RESP_END_MS, GUARD_MS = 10, 8.0, None, 1.0
SNR_ON, MIN_SNR = "median", 1.2
ANCHOR, ANCHOR_WIN_MS = "mean", 3.0
MAX_EDGE_FRAC, EDGE_MS, JITTER_MS = 0.5, 1.0, 0.5
KW = dict(n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, resp_end_ms=RESP_END_MS,
          guard_ms=GUARD_MS, min_snr=MIN_SNR, max_edge_frac=MAX_EDGE_FRAC, snr_on=SNR_ON,
          anchor=ANCHOR, anchor_win_ms=ANCHOR_WIN_MS)
CONSECUTIVE = 2        # intensities the response criterion must hold for

for k, f in RUN.items():
    meta, _, sig = load_run(f)
    have = {pretty(c) for c in sig if c != "Trigger A"}
    amps = sorted({m["amp_ma"] for m in meta})
    print(f"{k[0]:4s} {NAME[k[1]]:12s} elec={meta[0]['electrode']}  "
          f"{amps[0]}-{amps[-1]} mA ({len(amps)} steps)  "
          f"missing: {sorted(set(MUSCLES) - have) or 'none'}")

## 2 · Motor thresholds

Detected first, then corrected by hand. **One cell per recording**: run it, move each muscle's
slider to the lowest trace that carries a real response, press **Save**. A saved
`results/mt_<recording>.csv` replaces the detected values, so re-run the detection cell after
saving and everything below follows.

In [ ]:
MT = {}      # MT[(subject, protocol)] = {muscle: mA}
for s in SUBJECTS:
    for p in PROTOCOLS:
        csv = RUN[(s, p)]
        got = motor_thresholds([dict(label=s, csv=csv)], MUSCLES, consecutive=CONSECUTIVE,
                               verbose=False, **KW)[s]
        if os.path.exists(mt_file(csv)):          # hand-picked wins
            got = load_threshold_csv(mt_file(csv))
            print(f"hand-picked: {s} {NAME[p]}")
        MT[(s, p)] = got

print(f"\nmotor threshold (mA)\n{'muscle':22s}"
      + "".join(f"{s + ' ' + p[:5]:>14s}" for s in SUBJECTS for p in PROTOCOLS))
for m in MUSCLES:
    print(f"{m:22s}" + "".join(f"{str(MT[(s, p)].get(m) or '-'):>14s}"
                               for s in SUBJECTS for p in PROTOCOLS))

In [ ]:
def pick(key):
    """Open the picker for one recording, with a Save button. Re-run to correct."""
    csv = RUN[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  picks reloaded" if prev else ""))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev,
                            suggest=MT[key], key=csv,
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
pick(("P03", "burst"))

In [ ]:
pick(("P03", "arcex"))

In [ ]:
pick(("P04", "burst"))

In [ ]:
pick(("P04", "arcex"))

In [ ]:
pick(("NTA", "burst"))

In [ ]:
pick(("NTA", "arcex"))

## 3 · Figure 1 — how much more current does ARC-EX need?

A ratio per muscle per participant, so nothing participant-specific survives. **1.0 = the two
protocols recruit that muscle at the same current.**

In [ ]:
ratios = fig_threshold_ratio(MT, SUBJECTS, MUSCLES, protocols=("burst", "arcex"),
                             colours=SCOL, names=NAME,
                             title="ARC-EX vs 30 Hz burst - motor threshold ratio, baseline");

## 4 · Figure 2 — the shape of the recruitment

Both axes normalised inside each participant: x is intensity as a multiple of *that muscle's*
threshold, y is the response as % of *that muscle's* biggest. Every curve therefore crosses 1.0
at its own threshold and reaches 100 % at its own maximum — what is left to compare is the
**steepness**.

In [ ]:
CURVES = {k: recruitment(RUN[k], MUSCLES, MT[k], **KW) for k in RUN}
fig_recruitment(CURVES, MUSCLES, SUBJECTS, protocols=("burst", "arcex"), colours=SCOL,
                names=NAME, xmax=2.0,
                title="recruitment on a common axis - solid 30 Hz, dashed ARC-EX");

## 5 · Figure 3 — selectivity

As the current is raised above the **lowest** threshold of that recording, how much of the arm
comes in? A selective protocol adds muscles slowly; a broad one recruits most of them at once.

In [ ]:
sel = fig_selectivity(MT, SUBJECTS, MUSCLES, protocols=("burst", "arcex"),
                      levels=(1.0, 1.2, 1.5), names=NAME,
                      title="how many muscles are recruited above the lowest threshold");

## 6 · Figure 4 — depression along the train

2nd pulse and mean of pulses 2–10, each as % of that condition's own 1st pulse. Run **one
intensity step above** threshold, not at it: right at threshold the train fires intermittently
(a pulse responds, the next does not), which makes a pulse-to-pulse ratio close to a coin flip.

In [ ]:
DEP_STEPS = 1       # intensities above threshold; 0 = at threshold (unstable, see above)

tables = {}
for s in SUBJECTS:
    specs = []
    for p in PROTOCOLS:
        csv = RUN[(s, p)]
        amp = step_up(MT[(s, p)], csv, DEP_STEPS, verbose=False) if DEP_STEPS else MT[(s, p)]
        specs.append(dict(label=NAME[p], csv=csv, amp=amp,
                          colour=("#1f3b73" if p == "burst" else "#e6550d"),
                          hatch=("" if p == "burst" else "///")))
    use = [m for m in MUSCLES if all(m in sp["amp"] for sp in specs)]
    if not use:
        print(f"{s}: no muscle has a threshold in both protocols - skipped\n"); continue
    tables[s] = fig_depression(specs, use, metric="ratio", with_p1=True,
                               title=f"{s} - 30 Hz vs ARC-EX, each muscle at its MT"
                                     + (f"+{DEP_STEPS}" if DEP_STEPS else ""), **KW)
    print()

### The same numbers as one table

One row per participant × protocol × muscle, so the three can be pooled or plotted elsewhere.

In [ ]:
import pandas as pd
if tables:
    df = pd.concat({s: pd.DataFrame(t) for s, t in tables.items()},
                   names=["subject"]).reset_index(level=0)
    pd.set_option("display.width", 170, "display.max_columns", 20)
    print(df.round(3).to_string(index=False))